# Gate 3 -- pooled-vs-token campaign (primary decision)

Route B, Gate 3 (lock `gate2b_lock_2026-07-22.md`, D-030). Trains the primary
causal pair -- **identical** frozen tokens/text, identical rank-96 residual,
identical loss/batches/negatives/optimizer/checkpoint rule; the **only** difference
is scoring (pooled cosine vs token MaxSim) -- across 5 folds x 3 seeds x 2 arms on
the frozen 24-way pools, and computes the primary decision inputs.

**No GPU model to load:** it trains tiny adapters on already-extracted GLIM
representations. Attach: the **EEG token** dataset, the **batch-64 text token**
dataset, and the **P4b protocol** dataset (`candidate_pools.csv`,
`confirmation_donors.csv`, `outer_split_assignments.csv`). GPU + Internet +
`GITHUB_TOKEN`. Held-out test is never touched.

Reports the headline MaxSim macro-MRR and the Delta two-way lower bound vs the
locked **delta_sup = 0.02**. The three extended conditions (subject-held-out 16.3,
pool-size sweep 16.4, shuffled 9) are a follow-up run using `token_extended_pools`.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = '1477db88f116b2156e7eb65b9f19f1832618fe9e'
WORKTREE = '/kaggle/working/SemKey'
SEEDS = [20260722, 20260723, 20260724]      # three fixed seeds (lock 4/17)
EPOCHS, BATCH_SIZE, LR, TEMPERATURE, GRAD_CLIP = 40, 64, 1e-3, 0.05, 1.0   # lock 17
SELECT_EVERY = 80                           # dev-MRR checkpoint eval interval (steps)
POOL_SIZE = 24
ANALYTIC_CHANCE_MRR = 0.15733159073974598
DELTA_SUP = 0.02
assert len(COMMIT) == 40 and len(SEEDS) == 3

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, torch
from pathlib import Path
from kaggle_secrets import UserSecretsClient
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0)})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as h:
    h.write("#!/usr/bin/env python3\nimport os, sys\np = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in p else 'x-access-token')\n")
os.chmod(askpass, 0o700)
cenv = os.environ.copy(); cenv.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE): shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=cenv)
finally:
    os.remove(askpass); del github_token, cenv
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
assert subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip() == COMMIT
env = os.environ.copy(); env['PYTHONPATH'] = WORKTREE
for t in ('evaluation.test_token_campaign_driver', 'evaluation.test_token_decision', 'evaluation.test_primary_pair_eval'):
    subprocess.run([sys.executable, '-B', '-m', 'unittest', t], check=True, cwd=WORKTREE, env=env)
sys.path.insert(0, WORKTREE)
print({'clone': 'PASS', 'self_tests': 'PASS'})

In [ ]:
def find_one(pattern):
    hits = glob.glob('/kaggle/input/**/' + pattern, recursive=True)
    assert len(hits) == 1, ('need exactly one ' + pattern, hits)
    return hits[0]

def find_text_dir():
    # A stale text index may be bundled inside the EEG token dataset; prefer the
    # STANDALONE text-token dir (no EEG token_index.json beside it).
    dirs = [Path(os.path.dirname(h)) for h in glob.glob('/kaggle/input/**/text_token_index.csv', recursive=True)]
    if len(dirs) == 1:
        return dirs[0]
    standalone = [d for d in dirs if not glob.glob(str(d.parent) + '/**/token_index.json', recursive=True)]
    assert len(standalone) == 1, ('cannot disambiguate text_token_index.csv', [str(d) for d in dirs])
    return standalone[0]

eeg_root = Path(os.path.dirname(find_one('token_index.json')))
text_root = find_text_dir()
protocol_root = Path(os.path.dirname(find_one('candidate_pools.csv')))
# Confirm the text tokens are the batch-64 re-extraction (identity-verified).
_tman = json.load(open(text_root / 'text_token_manifest.json', encoding='utf-8'))
print({'eeg_root': str(eeg_root), 'text_root': str(text_root), 'protocol_root': str(protocol_root),
       'text_combined_chunk_sha256': _tman.get('combined_chunk_sha256')})

In [ ]:
from evaluation.token_campaign_io import (
    load_eeg_lookup, load_text_lookup, load_assignments, load_candidate_pools, load_donors)
eeg = load_eeg_lookup(eeg_root)
text = load_text_lookup(text_root)
assignments = load_assignments(protocol_root / 'outer_split_assignments.csv')
pools = load_candidate_pools(protocol_root / 'candidate_pools.csv')
donors = load_donors(protocol_root / 'confirmation_donors.csv')
folds = sorted({a['outer_fold'] for a in assignments})
print({'eeg_trials': len(eeg), 'texts': len(text), 'assignments': len(assignments), 'folds': folds})

# Put only the small unique-text cache on the GPU (~0.5 GB): candidate pools are
# stacked from it at scoring time, so the big candidate tensors never leave the GPU.
# EEG stays on CPU (~3.6 GB); its small per-trial query is moved to GPU on demand.
dev = torch.device('cuda')
text = {k: {kk: vv.to(dev) for kk, vv in v.items()} for k, v in text.items()}
print({'text_cache_on': str(dev), 'gpu_mem_gb': round(torch.cuda.memory_allocated() / 1e9, 2)})

In [ ]:
from evaluation.token_campaign_driver import run_campaign, aggregate_primary
from evaluation.token_training import TrainConfig
config = TrainConfig(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                     temperature=TEMPERATURE, grad_clip=GRAD_CLIP)
records, labels = run_campaign(
    folds, SEEDS, assignments, pools, donors, eeg, text,
    config=config, select_every=SELECT_EVERY, device='cuda', pool_size=POOL_SIZE)
result = aggregate_primary(records, labels, SEEDS)
print('n_trials:', result['n_trials'])

In [ ]:
summary, mrr, top1 = result['summary'], result['mrr_delta'], result['top1_delta']
report = {
    'project_commit': COMMIT, 'seeds': SEEDS, 'delta_sup': DELTA_SUP,
    'analytic_chance_mrr': ANALYTIC_CHANCE_MRR,
    'maxsim_correct_macro_mrr': summary['maxsim_correct_macro_mrr'],
    'maxsim_matched_wrong_macro_mrr': summary['maxsim_matched_wrong_macro_mrr'],
    'mean_collapse_gap': summary['mean_collapse_gap'],
    'mrr_delta_point': mrr['point'], 'mrr_delta_two_way_lower_bound': mrr['two_way_lower_bound'],
    'top1_delta_point': top1['point'], 'top1_delta_two_way_lower_bound': top1['two_way_lower_bound'],
    'seed_consistent': result['seed_consistent'], 'per_seed_delta': result['per_seed_delta'],
    'controls': result['controls'],
    'primary_conditions': {
        'mrr_lower_bound_ge_delta_sup': mrr['two_way_lower_bound'] >= DELTA_SUP,
        'both_metrics_lower_bound_positive': mrr['two_way_lower_bound'] > 0 and top1['two_way_lower_bound'] > 0,
        'seed_consistent': result['seed_consistent'],
        'mean_collapse_gap_positive': result['controls']['mean_collapse_gap_positive'],
        'matched_wrong_at_chance': result['controls']['matched_wrong_at_chance'],
    },
    'note': ('PRIMARY decision inputs at pool 24. Full win also requires the three '
             'extended conditions (subject-held-out 16.3, pool-size sweep 16.4, '
             'shuffled 9) from the follow-up run; a null is reported, not reframed.'),
}
out = '/kaggle/working/pooled_vs_token_primary_report.json'
with open(out, 'w', encoding='utf-8') as h:
    json.dump(report, h, indent=2, sort_keys=True); h.write('\n')
report_sha = hashlib.sha256(open(out, 'rb').read()).hexdigest()
for p in (WORKTREE,):
    pass  # keep WORKTREE for provenance
print('=== POOLED-vs-TOKEN PRIMARY RESULT ===')
print('MaxSim correct macro-MRR :', round(summary['maxsim_correct_macro_mrr'], 4),
      '(pooled arm anchor ~0.32; analytic chance', round(ANALYTIC_CHANCE_MRR, 4), ')')
print('matched-wrong macro-MRR  :', round(summary['maxsim_matched_wrong_macro_mrr'], 4))
print('Delta (MaxSim - pooled)   : point', round(mrr['point'], 4),
      '| two-way lower bound', round(mrr['two_way_lower_bound'], 4), '| delta_sup', DELTA_SUP)
print('Top-1 Delta               : point', round(top1['point'], 4),
      '| two-way lower bound', round(top1['two_way_lower_bound'], 4))
print('primary conditions        :', report['primary_conditions'])
print('report sha256             :', report_sha)
print('saved', out)

Save `pooled_vs_token_primary_report.json` as a private dataset. Then report the
`Delta ... two-way lower bound` vs `delta_sup = 0.02`, `seed_consistent`, and the
two controls. If the primary conditions hold, run the extended-conditions follow-up
(subject-held-out, pool-size sweep, shuffled) before declaring a full win. Either
way the result is reported as-is; a null is not reframed (lock 13). Held-out test
stays sealed.